# LexisNexis Fraud Detection Project

Break Through Tech AI Studio - Fall 2026

This notebook contains advanced feature engineering using the cleaned dataset from M1.

Task 1 Feature Engineering and Preprocessing Summary: We carried forward the 26 features from September and created additional features describing age groups, approximate vehicle age, claim reporting lag, and weekend timing. We also grouped infrequent levels of higher-cardinality categorical variables during preprocessing. The preprocessing pipeline performs median imputation and scaling for numeric variables and one-hot encoding for categorical variables. All learned preprocessing parameters, including medians, scaling parameters, and rare-category groupings, are fit using the 1994 training set only and then applied unchanged to the 1995 validation and 1996 test sets.


In [1]:
import pandas as pd
import numpy as np

!rm -rf LexisNexis-1A-detecting-fraudulent-insurance-claims
!git clone https://github.com/Break-Through-Tech/LexisNexis-1A-detecting-fraudulent-insurance-claims.git

%cd /content/LexisNexis-1A-detecting-fraudulent-insurance-claims

df = pd.read_csv("data/fraud_oracle_cleaned.csv")

TARGET = "FraudFound_P"

SAFE_FEATURES = [
    "Month",
    "WeekOfMonth",
    "DayOfWeek",
    "MonthClaimed",
    "WeekOfMonthClaimed",
    "DayOfWeekClaimed",
    "Make",
    "VehicleCategory",
    "VehiclePrice",
    "AgeOfVehicle",
    "NumberOfCars",
    "AccidentArea",
    "Fault",
    "Sex",
    "MaritalStatus",
    "Age",
    "Age_missing",
    "AgeOfPolicyHolder",
    "PolicyType",
    "BasePolicy",
    "Deductible",
    "DriverRating",
    "AgentType",
    "Days_Policy_Accident",
    "Days_Policy_Claim",
    "PastNumberOfClaims"
]

Cloning into 'LexisNexis-1A-detecting-fraudulent-insurance-claims'...
remote: Enumerating objects: 179, done.
remote: Counting objects: 100% (179/179), done.
remote: Compressing objects: 100% (177/177), done.
remote: Total 179 (delta 103), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (179/179), 1.49 MiB | 3.80 MiB/s, done.
Resolving deltas: 100% (103/103), done.
/content/LexisNexis-1A-detecting-fraudulent-insurance-claims


In [2]:
train = df[df["Year"] == 1994].copy()
val   = df[df["Year"] == 1995].copy()
test  = df[df["Year"] == 1996].copy()

print("Train:", train.shape)
print("Validation:", val.shape)
print("Test:", test.shape)

Train: (6141, 34)
Validation: (5195, 34)
Test: (4083, 34)


##Here is feature engineering:

In [3]:
MONTH_NUM = {
    "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4,
    "May": 5, "Jun": 6, "Jul": 7, "Aug": 8,
    "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12
}

VEHICLE_AGE_MAP = {
    "new": 0,
    "2 years": 2,
    "3 years": 3,
    "4 years": 4,
    "5 years": 5,
    "6 years": 6,
    "7 years": 7,
    "more than 7": 8
}


def add_engineered_features(data):
    data = data.copy()

    # Age groups derived from the cleaned numeric Age field
    data["AgeBand"] = pd.cut(
        data["Age"],
        bins=[0, 24, 34, 44, 54, 64, np.inf],
        labels=["<25", "25-34", "35-44", "45-54", "55-64", "65+"]
    )

    # Preserve missing Age explicitly rather than assigning it an age band
    data["AgeBand"] = data["AgeBand"].astype("object").fillna("Missing")

    # Convert vehicle-age bucket into an approximate ordered numeric feature
    data["VehicleAgeApprox"] = data["AgeOfVehicle"].map(VEHICLE_AGE_MAP)

    # Approximate number of months between accident and claim filing
    accident_month = data["Month"].map(MONTH_NUM)
    claimed_month = data["MonthClaimed"].map(MONTH_NUM)

    data["ClaimLagMonths"] = (
        claimed_month - accident_month
    ) % 12

    # Policy was under 30 days old when the accident happened.
    # "none" is included because it sits with the other short-duration buckets
    # and carries the highest fraud rate in the column (9 of 55 claims).
    data["NewPolicy"] = (
        data["Days_Policy_Accident"]
        .isin(["none", "1 to 7", "8 to 15", "15 to 30"])
    ).astype(int)

    # Simple timing features
    data["WeekendAccident"] = (
        data["DayOfWeek"].isin(["Saturday", "Sunday"])
    ).astype(int)

    data["WeekendClaim"] = (
        data["DayOfWeekClaimed"].isin(["Saturday", "Sunday"])
    ).astype(int)

    return data

In [4]:
X_train = add_engineered_features(train[SAFE_FEATURES])
X_val   = add_engineered_features(val[SAFE_FEATURES])
X_test  = add_engineered_features(test[SAFE_FEATURES])

y_train = train[TARGET]
y_val   = val[TARGET]
y_test  = test[TARGET]

In [5]:
# --- sanity checks before we trust the engineered features ---

# 1. did every value actually find a match in the two maps
print("AgeOfVehicle values missing from the map:",
      set(df["AgeOfVehicle"].unique()) - set(VEHICLE_AGE_MAP.keys()))
print("Month values missing from the map:",
      set(df["Month"].unique()) - set(MONTH_NUM.keys()))
print("MonthClaimed values missing from the map:",
      set(df["MonthClaimed"].unique()) - set(MONTH_NUM.keys()))

# 2. did anything come out blank after mapping
for name, X in [("train", X_train), ("val", X_val), ("test", X_test)]:
    print(f"\n{name} blanks after mapping:")
    print(X[["VehicleAgeApprox", "ClaimLagMonths"]].isna().sum())

# 3. what does the claim lag actually look like
print("\nClaimLagMonths spread in training:")
print(X_train["ClaimLagMonths"].value_counts().sort_index())

# 4. what "none" means in Days_Policy_Accident, which NewPolicy depends on
print("\nDays_Policy_Accident breakdown:")
print(df.groupby("Days_Policy_Accident")[TARGET].agg(
    claims="count", fraud="sum", rate="mean"))

# 5. are the weekend flags actually catching anything
print("\nWeekend flag counts in training:")
print(X_train[["WeekendAccident", "WeekendClaim"]].sum())

AgeOfVehicle values missing from the map: set()
Month values missing from the map: set()
MonthClaimed values missing from the map: set()

train blanks after mapping:
VehicleAgeApprox    0
ClaimLagMonths      0
dtype: int64

val blanks after mapping:
VehicleAgeApprox    0
ClaimLagMonths      0
dtype: int64

test blanks after mapping:
VehicleAgeApprox    0
ClaimLagMonths      0
dtype: int64

ClaimLagMonths spread in training:
ClaimLagMonths
0     4515
1     1323
2      149
3       65
4       33
5       14
6       16
7       11
8        5
9        5
10       1
11       4
Name: count, dtype: int64

Days_Policy_Accident breakdown:
                      claims  fraud      rate
Days_Policy_Accident                         
1 to 7                    14      1  0.071429
15 to 30                  49      3  0.061224
8 to 15                   55      5  0.090909
more than 30           15246    905  0.059360
none                      55      9  0.163636

Weekend flag counts in training:
WeekendAcc

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [7]:
numeric_features = (
    X_train.select_dtypes(include="number")
    .columns
    .tolist()
)

categorical_features = (
    X_train.select_dtypes(exclude="number")
    .columns
    .tolist()
)

In [8]:
grouped_categories = [
    "Make",
    "PolicyType"
]

regular_categories = [
    col for col in categorical_features
    if col not in grouped_categories
]

In [9]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [10]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

In [11]:
grouped_category_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=0.01
    ))
])

In [12]:
preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        numeric_features
    ),
    (
        "categorical",
        categorical_pipeline,
        regular_categories
    ),
    (
        "grouped_categorical",
        grouped_category_pipeline,
        grouped_categories
    )
])

In [13]:
preprocessor.fit(X_train)

ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['WeekOfMonth', 'WeekOfMonthClaimed', 'Age',
                                  'Age_missing', 'Deductible', 'DriverRating',
                                  'VehicleAgeApprox', 'ClaimLagMonths',
                                  'NewPolicy', 'WeekendAccident',
                                  'WeekendClaim']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleI...
                                  'NumberOfCars', 'AccidentArea', 'Fault',
                                  'Sex', 'MaritalStatus', 'AgeOfPolicyHolder',
                                  'BasePolicy', 'AgentType',
                                  'Days_Policy_Accident', 'Days_Policy_Claim',
                                  'PastNumberOfClaims', 'AgeBand']),
                                ('grouped_categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist',
                                                                min_frequency=0.01))]),
                                 ['Make', 'PolicyType'])])

In [14]:
X_train_processed = preprocessor.transform(X_train)
X_val_processed   = preprocessor.transform(X_val)
X_test_processed  = preprocessor.transform(X_test)

In [15]:
print("Raw train shape:", X_train.shape)
print("Processed train shape:", X_train_processed.shape)

print("Processed validation shape:", X_val_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Raw train shape: (6141, 32)
Processed train shape: (6141, 129)
Processed validation shape: (5195, 129)
Processed test shape: (4083, 129)


In [16]:
# Save the engineered dataset so Tasks 2 to 5 all start from identical features
cols = SAFE_FEATURES + ["Year", TARGET]
engineered = add_engineered_features(df[cols])
engineered.to_csv("fraud_oracle_engineered.csv", index=False)

print("Saved fraud_oracle_engineered.csv")
print("Shape:", engineered.shape)
print("\nColumns:")
for c in engineered.columns:
    print(" -", c)

Saved fraud_oracle_engineered.csv
Shape: (15419, 34)

Columns:
 - Month
 - WeekOfMonth
 - DayOfWeek
 - MonthClaimed
 - WeekOfMonthClaimed
 - DayOfWeekClaimed
 - Make
 - VehicleCategory
 - VehiclePrice
 - AgeOfVehicle
 - NumberOfCars
 - AccidentArea
 - Fault
 - Sex
 - MaritalStatus
 - Age
 - Age_missing
 - AgeOfPolicyHolder
 - PolicyType
 - BasePolicy
 - Deductible
 - DriverRating
 - AgentType
 - Days_Policy_Accident
 - Days_Policy_Claim
 - PastNumberOfClaims
 - Year
 - FraudFound_P
 - AgeBand
 - VehicleAgeApprox
 - ClaimLagMonths
 - NewPolicy
 - WeekendAccident
 - WeekendClaim


In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

# one engineered copy of the data, split the same way as the rest of the notebook
eng = add_engineered_features(df[SAFE_FEATURES + ["Year", TARGET]])
tr = eng[eng["Year"] == 1994]
va = eng[eng["Year"] == 1995]

ENGINEERED = ["AgeBand", "VehicleAgeApprox", "ClaimLagMonths",
              "NewPolicy", "WeekendAccident", "WeekendClaim"]

# the original columns each new feature was built out of
SOURCES = ["Age", "AgeOfVehicle", "Month", "MonthClaimed",
           "DayOfWeek", "DayOfWeekClaimed", "Days_Policy_Accident"]

feature_sets = {
    "A. September 26 only": SAFE_FEATURES,
    "B. 26 + all 6 new (current)": SAFE_FEATURES + ENGINEERED,
    "C. New ones replace sources": (
        [c for c in SAFE_FEATURES if c not in SOURCES] + ENGINEERED
    ),
}


def build_model(X):
    num = X.select_dtypes(include="number").columns.tolist()
    cat = X.select_dtypes(exclude="number").columns.tolist()
    pre = ColumnTransformer([
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler())
        ]), num),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore"))
        ]), cat),
    ])
    return Pipeline([("pre", pre),
                     ("clf", LogisticRegression(max_iter=1000))])


def capture_at(y_true, scores, pct):
    """share of all the fraud that lands in the top pct of scored claims"""
    n = int(len(scores) * pct)
    top = np.argsort(scores)[::-1][:n]
    return y_true.iloc[top].sum() / y_true.sum()


results = {}
for name, feats in feature_sets.items():
    m = build_model(tr[feats]).fit(tr[feats], tr[TARGET])
    p = m.predict_proba(va[feats])[:, 1]
    results[name] = {
        "features": len(feats),
        "PR-AUC": average_precision_score(va[TARGET], p),
        "Fraud caught, top 10%": capture_at(va[TARGET], p, 0.10),
        "Fraud caught, top 20%": capture_at(va[TARGET], p, 0.20),
    }

print("Trained on 1994, scored on 1995. The 1996 test year is not touched.\n")
display(pd.DataFrame(results).T.round(4))

Trained on 1994, scored on 1995. The 1996 test year is not touched.



,features,PR-AUC,"Fraud caught, top 10%","Fraud caught, top 20%"
A. September 26 only,26.0,0.1002,0.1761,0.3488
B. 26 + all 6 new (current),32.0,0.0979,0.1595,0.3522
C. New ones replace sources,25.0,0.1460,0.2658,0.5216


In [18]:
# which part of the swap is doing the work
SWAPS = {
    "vehicle age": (["AgeOfVehicle"], ["VehicleAgeApprox"]),
    "months": (["Month", "MonthClaimed"], ["ClaimLagMonths"]),
    "weekdays": (["DayOfWeek", "DayOfWeekClaimed"],
                 ["WeekendAccident", "WeekendClaim"]),
    "policy age": (["Days_Policy_Accident"], ["NewPolicy"]),
    "claimant age": (["Age"], ["AgeBand"]),
}

sets = {
    "A. September 26 (reference)": SAFE_FEATURES,
    "C. all swaps (reference)": (
        [c for c in SAFE_FEATURES if c not in SOURCES] + ENGINEERED
    ),
    "D. drop the 7 sources, add nothing back": (
        [c for c in SAFE_FEATURES if c not in SOURCES]
    ),
}

for label, (drop_these, add_these) in SWAPS.items():
    sets[f"swap {label} only"] = (
        [c for c in SAFE_FEATURES if c not in drop_these] + add_these
    )

rows = {}
for name, feats in sets.items():
    m = build_model(tr[feats]).fit(tr[feats], tr[TARGET])
    p = m.predict_proba(va[feats])[:, 1]
    rows[name] = {
        "features": len(feats),
        "encoded cols": len(m.named_steps["pre"].get_feature_names_out()),
        "PR-AUC": average_precision_score(va[TARGET], p),
        "top 10%": capture_at(va[TARGET], p, 0.10),
        "top 20%": capture_at(va[TARGET], p, 0.20),
    }

print("All trained on 1994, scored on 1995.\n")
display(pd.DataFrame(rows).T.round(4).sort_values("PR-AUC", ascending=False))

All trained on 1994, scored on 1995.



,features,encoded cols,PR-AUC,top 10%,top 20%
swap months only,25.0,105.0,0.1491,0.2890,0.5282
"D. drop the 7 sources, add nothing back",19.0,76.0,0.1485,0.2957,0.5415
C. all swaps (reference),25.0,88.0,0.1460,0.2658,0.5216
swap policy age only,26.0,124.0,0.1016,0.1694,0.3522
swap claimant age only,26.0,134.0,0.1003,0.1661,0.3455
A. September 26 (reference),26.0,128.0,0.1002,0.1761,0.3488
swap weekdays only,26.0,116.0,0.0993,0.1794,0.3389
swap vehicle age only,26.0,121.0,0.0990,0.1661,0.3389


In [19]:
NO_MONTHS = [c for c in SAFE_FEATURES if c not in ["Month", "MonthClaimed"]]
MINIMAL   = [c for c in SAFE_FEATURES if c not in SOURCES]

more_sets = {
    "A. September 26 (reference)": SAFE_FEATURES,
    "D. drop all 7 sources (reference)": MINIMAL,
    "E. drop Month + MonthClaimed only": NO_MONTHS,
    "F. D plus ClaimLagMonths": MINIMAL + ["ClaimLagMonths"],
    "G. E plus ClaimLagMonths": NO_MONTHS + ["ClaimLagMonths"],
}

rows2 = {}
for name, feats in more_sets.items():
    m = build_model(tr[feats]).fit(tr[feats], tr[TARGET])
    p = m.predict_proba(va[feats])[:, 1]
    rows2[name] = {
        "features": len(feats),
        "encoded cols": len(m.named_steps["pre"].get_feature_names_out()),
        "PR-AUC": average_precision_score(va[TARGET], p),
        "top 10%": capture_at(va[TARGET], p, 0.10),
        "top 20%": capture_at(va[TARGET], p, 0.20),
    }

print("Trained on 1994, scored on 1995.\n")
display(pd.DataFrame(rows2).T.round(4).sort_values("PR-AUC", ascending=False))

Trained on 1994, scored on 1995.



,features,encoded cols,PR-AUC,top 10%,top 20%
E. drop Month + MonthClaimed only,24.0,104.0,0.1491,0.2890,0.5316
G. E plus ClaimLagMonths,25.0,105.0,0.1491,0.2890,0.5282
D. drop all 7 sources (reference),19.0,76.0,0.1485,0.2957,0.5415
F. D plus ClaimLagMonths,20.0,77.0,0.1472,0.2990,0.5349
A. September 26 (reference),26.0,128.0,0.1002,0.1761,0.3488


In [20]:
# ---------------------------------------------------------------
# FINAL FEATURE LIST FOR MILESTONE 2
# Tasks 2 through 6 should all use this exact list.
#
# It is the 26 features from September, minus Month and MonthClaimed.
# Those two one-hot encode into 24 columns, and with only 409 fraud
# cases in the 1994 training year the model fits noise in them that
# does not carry over to 1995. Dropping them moved validation PR-AUC
# from 0.1002 to 0.1491 and top 20% capture from 34.9% to 53.2%.
#
# None of the six engineered features improved validation, so none of
# them are in this list. They are still saved in
# fraud_oracle_engineered.csv in case anyone wants to retest them with
# a tree model, which handles these columns differently.
# ---------------------------------------------------------------

MODEL_FEATURES = [c for c in SAFE_FEATURES
                  if c not in ["Month", "MonthClaimed"]]

print("Milestone 2 feature list:", len(MODEL_FEATURES), "features\n")
for c in MODEL_FEATURES:
    print(" -", c)

Milestone 2 feature list: 24 features

 - WeekOfMonth
 - DayOfWeek
 - WeekOfMonthClaimed
 - DayOfWeekClaimed
 - Make
 - VehicleCategory
 - VehiclePrice
 - AgeOfVehicle
 - NumberOfCars
 - AccidentArea
 - Fault
 - Sex
 - MaritalStatus
 - Age
 - Age_missing
 - AgeOfPolicyHolder
 - PolicyType
 - BasePolicy
 - Deductible
 - DriverRating
 - AgentType
 - Days_Policy_Accident
 - Days_Policy_Claim
 - PastNumberOfClaims
